In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi
from shapely.geometry import Polygon, Point
from shapely.ops import unary_union

def gerar_estudo_voronoi_sp(caminho_shapefile=None, raio_km=0.7):

    print("1. Carregando a malha territorial de São Paulo...")
    
    # Carrega do servidor de rede ou faz o download automático se não encontrar
    if caminho_shapefile and os.path.exists(caminho_shapefile):
        cidades_sp = gpd.read_file(caminho_shapefile)
    else:
        print("Caminho local não encontrado ou não fornecido. Baixando via API do IBGE (Malha 2022)...")
        url_ibge = "https://ibge.gov.br"
        cidades_sp = gpd.read_file(url_ibge)
    
    # Padronizar nomes das colunas para maiúsculas
    cidades_sp.columns = [col.upper() for col in cidades_sp.columns]
    
    # Define explicitamente a coluna GEOMETRY ativa para evitar o AttributeError
    if "GEOMETRY" in cidades_sp.columns:
        cidades_sp = cidades_sp.set_geometry("GEOMETRY")
    
    # Identificar a coluna de nomes do município
    col_nome = "NM_MUN" if "NM_MUN" in cidades_sp.columns else "NM_MUNICIP"
    sp_poly = cidades_sp[cidades_sp[col_nome].str.upper() == "SÃO PAULO"]
    
    if sp_poly.empty:
        raise ValueError("Município de São Paulo não encontrado no Shapefile fornecido.")
    
    # Conversão de Projeção para SIRGAS 2000 / UTM zone 23S (EPSG:31983) para trabalhar em metros
    sp_poly = sp_poly.to_crs(epsg=31983)
    poligono_sp = sp_poly.geometry.iloc[0]
    
    print(f"Área oficial calculada de SP: {poligono_sp.area / 1e6:.2f} km²")
    
    # 2. Geração da grade hexagonal de pontos (distância ideal de 2 * raio entre os centros)
    print(f"\n2. Gerando grade de pontos com raio de {raio_km} km...")
    distancia_pontos = raio_km * 2 * 1000  # Convertendo quilômetros para metros
    
    minx, miny, maxx, maxy = poligono_sp.bounds
    
    dx = distancia_pontos
    dy = distancia_pontos * np.sqrt(3) / 2
    
    pontos_validos = []
    
    # Loop para cobrir o bounding box de SP criando a colmeia hexagonal
    for i, y in enumerate(np.arange(miny - dy, maxy + dy, dy)):
        offset = (dx / 2) if i % 2 == 1 else 0
        for x in np.arange(minx - dx + offset, maxx + dx, dx):
            p = Point(x, y)
            # Critério restrito: O ponto e seu raio estipulado devem estar totalmente contidos em SP
            if p.buffer(raio_km * 1000).within(poligono_sp):
                pontos_validos.append(p)
                
    total_pontos = len(pontos_validos)
    print(f"-> Foram posicionados {total_pontos} pontos totalmente dentro das fronteiras.")
    
    if total_pontos < 4:
        print("Erro: Poucos pontos gerados para estruturar um diagrama de Voronoi completo.")
        return
    
    # 3. Construindo o Diagrama de Voronoi
    print("\n3. Construindo polígonos de Voronoi...")
    coords = np.array([[p.x, p.y] for p in pontos_validos])
    
    # Adicionando pontos dummy distantes para fechar as bordas abertas do Voronoi do SciPy
    margem_buffer = 100000  # 100 km de margem
    pontos_dummy = np.array([
        [minx - margem_buffer, miny - margem_buffer],
        [maxx + margem_buffer, miny - margem_buffer],
        [maxx + margem_buffer, maxy + margem_buffer],
        [minx - margem_buffer, maxy + margem_buffer]
    ])
    coords_totais = np.vstack([coords, pontos_dummy])
    
    vor = Voronoi(coords_totais)
    
    # Transformando as partições matemáticas em polígonos do Shapely recortados no limite de SP
    voronoi_polygons = []
    for i in range(total_pontos):
        region_idx = vor.point_region[i]
        v_vertices = vor.regions[region_idx]
        
        if -1 not in v_vertices and len(v_vertices) > 0:
            polygon_vertices = vor.vertices[v_vertices]
            vor_poly = Polygon(polygon_vertices)
            
            # Recorte exato da célula com o limite municipal de SP
            vor_poly_clipped = vor_poly.intersection(poligono_sp)
            if not vor_poly_clipped.is_empty:
                voronoi_polygons.append(vor_poly_clipped)
                
    # 4. Criando os GeoDataFrames finais na projeção UTM métrica (EPSG:31983)
    gdf_pontos_utm = gpd.GeoDataFrame(geometry=pontos_validos, crs=sp_poly.crs)
    gdf_pontos_utm['id_ponto'] = range(1, len(gdf_pontos_utm) + 1)
    
    gdf_voronoi_utm = gpd.GeoDataFrame(geometry=voronoi_polygons, crs=sp_poly.crs)
    gdf_voronoi_utm['id_celula'] = range(1, len(gdf_voronoi_utm) + 1)
    gdf_voronoi_utm['area_km2'] = gdf_voronoi_utm.geometry.area / 1e6
    
    # 5. Plotagem Gráfica com Matplotlib
    print("\n4. Gerando visualização do mapa...")
    fig, ax = plt.subplots(figsize=(10, 12))
    
    sp_poly.plot(ax=ax, color='none', edgecolor='black', linewidth=1.5, label='Limite SP')
    gdf_voronoi_utm.plot(ax=ax, cmap='tab20', alpha=0.6, edgecolor='white', linewidth=0.5)
    gdf_pontos_utm.plot(ax=ax, color='red', markersize=5, zorder=5, label=f'Centros (Raio {raio_km}km)')
    
    plt.title(f"Diagrama de Voronoi em SP\nFiltro Restrito: Centros com Raio de {raio_km}km 100% Contidos", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.xlabel("Easting (Metros - UTM 23S)")
    plt.ylabel("Northing (Metros - UTM 23S)")
    
    # 6. CONFIGURAÇÃO DO DIRETÓRIO DE DESTINO SOLICITADO
    print("\n5. Exportando resultados em formato Shapefile (.shp)...")
    
    # Definindo de forma absoluta o caminho solicitado
    pasta_raiz = r"C:\Users\A0050637\Documents\01_Diversos\FIAP\GS_2026_2sem"
    pasta_utm = os.path.join(pasta_raiz, "shapefiles_utm_31983")
    pasta_wgs = os.path.join(pasta_raiz, "shapefiles_wgs84_4326")
    
    os.makedirs(pasta_utm, exist_ok=True)
    os.makedirs(pasta_wgs, exist_ok=True)
    
    # Criando as versões alternativas projetadas em WGS84 (Graus Decimais)
    gdf_pontos_wgs84 = gdf_pontos_utm.to_crs(epsg=4326)
    gdf_voronoi_wgs84 = gdf_voronoi_utm.to_crs(epsg=4326)
    sp_bounds_wgs84 = sp_poly.to_crs(epsg=4326)
    
    # Salvando em formato ESRI Shapefile - Versão 1: UTM Métrico
    gdf_pontos_utm.to_file(os.path.join(pasta_utm, "pontos_centros_utm.shp"), driver="ESRI Shapefile")
    gdf_voronoi_utm.to_file(os.path.join(pasta_utm, "celulas_voronoi_utm.shp"), driver="ESRI Shapefile")
    sp_poly.to_file(os.path.join(pasta_utm, "limite_municipio_sp_utm.shp"), driver="ESRI Shapefile")
    
    # Salvando em formato ESRI Shapefile - Versão 2: WGS84 Geográfico
    gdf_pontos_wgs84.to_file(os.path.join(pasta_wgs, "pontos_centros_wgs84.shp"), driver="ESRI Shapefile")
    gdf_voronoi_wgs84.to_file(os.path.join(pasta_wgs, "celulas_voronoi_wgs84.shp"), driver="ESRI Shapefile")
    sp_bounds_wgs84.to_file(os.path.join(pasta_wgs, "limite_municipio_sp_wgs84.shp"), driver="ESRI Shapefile")
    
    print(f"\n=== ESTUDO CONCLUÍDO COM SUCESSO ===")
    print(f"Total de pontos internos mapeados: {total_pontos}")
    print(f"Destino dos Shapefiles UTM: '{pasta_utm}'")
    print(f"Destino dos Shapefiles WGS84: '{pasta_wgs}'")
    

if __name__ == "__main__":
    # Caminho do arquivo de malha de entrada no servidor
    CAMINHO_LOCAL = r"\\10.128.222.237\compartilhado\Paulo Pereira\Modelo_Novas_Cidades\sp_municipios\35MUE250GC_SIR.shp"
    
    # Executando a simulação padrão com raio_km=1.0
    gerar_estudo_voronoi_sp(caminho_shapefile=CAMINHO_LOCAL, raio_km=0.7)
